In [86]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()


True

In [87]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-20b",
    groq_api_key=os.environ.get("GROQ_API_KEY")
)

In [88]:
from langchain_core.messages import AIMessage, HumanMessage

messages = [
    AIMessage(content="Bạn là trợ lý AI"),
    HumanMessage(content="Python là gì"),
]

result = model.invoke(messages)

In [89]:
result.content

'Python là một ngôn ngữ lập trình cấp cao, đa năng và dễ học, được Guido van Rossum phát triển và ra mắt lần đầu vào năm 1991. Nó nổi bật với cú pháp rõ ràng, ngắn gọn và phong phú, giúp lập trình viên viết code nhanh chóng và dễ bảo trì.\n\n**Điểm nổi bật của Python:**\n\n| Tính năng | Mô tả |\n|-----------|-------|\n| **Đa nền tảng** | Chạy được trên Windows, macOS, Linux, và nhiều hệ điều hành khác. |\n| **Cú pháp trực quan** | Sử dụng dấu cách (indentation) để xác định khối lệnh, giảm bớt ký hiệu lệnh phức tạp. |\n| **Thư viện phong phú** | Hơn 200.000 gói thư viện (Python Package Index – PyPI) cho mọi mục đích: web, khoa học dữ liệu, trí tuệ nhân tạo, trò chơi, … |\n| **Tự động quản lý bộ nhớ** | Thuật toán thu gom rác (garbage collection) giúp giảm lỗi bộ nhớ. |\n| **Hỗ trợ lập trình hướng đối tượng và thủ tục** | Cho phép lập trình viên linh hoạt lựa chọn phong cách phù hợp. |\n| **Tương thích với C/C++** | Có thể nhúng code C/C++ để tối ưu hiệu suất khi cần. |\n\n**Các ứng dụng

In [90]:
# Message history/context
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

In [91]:
store = {} # Đây là dict lưu trữ các cuộc hội thoại, theo sesion (sesion ID - conversation ID)

def get_session_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory() # Khởi tạo một cuộc hội thoại mới nếu chưa có
    return store[session_id]

In [92]:
with_message_history = RunnableWithMessageHistory(model, get_session_history)

/Users/vunguyen/Documents/Mine/Python_Cyber/Lesson2/myenv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [93]:
chat_session_1 = {
    "configurable": {
        "session_id": "session_1"
    }
}

In [94]:
messages = [
    HumanMessage(content="Xin chào, tôi là Vũ, tôi là một software developer")
]
response = with_message_history.invoke(
    messages,
    config=chat_session_1
)

In [95]:
response.content

'Chào Vũ! Rất vui được gặp bạn. Bạn đang làm việc gì trong lĩnh vực phát triển phần mềm? Có câu hỏi, vấn đề hay dự án nào bạn muốn trao đổi, nhận tư vấn hay chỉ muốn chia sẻ kinh nghiệm không? Tôi luôn sẵn sàng hỗ trợ!'

In [96]:
chat_session_2 = {
    "configurable": {
        "session_id": "session_2"
    }
}

In [97]:
messages = [
    HumanMessage(content="Tôi là ai, và tôi làm gì?")
]
response = with_message_history.invoke(
    messages,
    config=chat_session_1
)

In [98]:
response.content

'Bạn là **Vũ**, một **software developer** – người viết, kiểm thử và bảo trì phần mềm.  \n- **Nghĩa vụ chính**: thiết kế kiến trúc ứng dụng, viết code (Java, Python, JavaScript, C#, v.v.), debug, tối ưu, và đảm bảo chất lượng bằng unit test, integration test.  \n- **Kỹ năng thường gặp**: làm việc với các framework, database, API, CI/CD, version control (Git) và hiểu biết về DevOps.  \n- **Mục tiêu**: tạo ra các giải pháp phần mềm hiệu quả, bảo trì dễ dàng và đáp ứng nhu cầu người dùng hoặc doanh nghiệp.  \n\nNếu bạn muốn chia sẻ thêm về lĩnh vực, công nghệ hay dự án cụ thể mà mình đang làm, mình rất sẵn lòng lắng nghe và hỗ trợ!'

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Bạn là trợ lý AI, bạn sẽ trả lời bằng {language}"),  # ✅ 1 chuỗi
        MessagesPlaceholder(variable_name="input")
    ]
)
chain = prompt | model

In [100]:
chain_message_history = RunnableWithMessageHistory(chain, get_session_history, input_messages_key="input")

/Users/vunguyen/Documents/Mine/Python_Cyber/Lesson2/myenv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [101]:
chat_session_new = {
    "configurable": {
        "session_id": "chat_session_new"
    }
}

In [102]:
response = chain_message_history.invoke(
    {
        "input": [
            HumanMessage(content="Python là gì?")
        ],
        "language": "Tiếng Việt"
    },
    config=chat_session_new
)

In [103]:
response.content

'Python là một ngôn ngữ lập trình đa năng, được thiết kế với cú pháp rõ ràng, dễ đọc và dễ viết. Nó được phát triển bởi Guido van Rossum và lần đầu ra mắt vào năm 1991. Python hỗ trợ nhiều mô hình lập trình, bao gồm lập trình hướng đối tượng, lập trình thủ tục và lập trình hàm. Nhờ vào thư viện chuẩn phong phú và cộng đồng phát triển lớn, Python được ứng dụng rộng rãi trong các lĩnh vực như phát triển web, khoa học dữ liệu, trí tuệ nhân tạo, tự động hóa, và nhiều hơn nữa.'